# CET433 Project 2: Explainable Network IDS for SOC Analysts

## Required project work only

**Customer:** Company SOC or Managed Security Service Provider (MSSP).  
**Problem:** Detect malicious network flows and provide understandable reasons for alerts.  
**Dataset:** UNSW-NB15 official training and testing CSV files.

This notebook focuses on the required Project 2 work: binary and multiclass IDS, preprocessing without leakage, baseline and tuned model comparison, per-class recall, SHAP explanations, robustness/distribution-shift evaluation, and saved model output.

**Important:** The instructor also asks for a SOC alerts dashboard and a random-vs-temporal split comparison. The supplied UNSW-NB15 partition files do not provide a usable timestamp column, so the official held-out test comparison below is a distribution-shift check, not a true temporal split. Do not describe it as temporal validation. A timestamped dataset or a separate dashboard app is still needed if these are required as separate deliverables.


In [ ]:
import sys
import os
import platform

print("Python version:", sys.version)
print("Operating system:", platform.system())
print("Working directory:", os.getcwd())

try:
    import google.colab
    print("✅ Google Colab runtime detected")
except ImportError:
    print("❌ Not running in Google Colab")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

print("Google Drive mounted:", drive_root.exists())

if drive_root.exists():
    print("Top-level folders:")
    for item in list(drive_root.iterdir())[:15]:
        print("-", item.name)

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Verify access
drive_root = Path("/content/drive/MyDrive")

print("Google Drive mounted:", drive_root.exists())

if drive_root.exists():
    print("\nTop-level folders:")
    for item in list(drive_root.iterdir())[:15]:
        print("-", item.name)
        

In [ ]:
from pathlib import Path

project_root = Path(
    "/content/drive/MyDrive/CET433_Explainable_IDS"
)

folders = [
    project_root / "data" / "official",
    project_root / "snapshots",
    project_root / "models",
    project_root / "results",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    print("Ready:", folder)

print("\nProject folder created successfully!")


In [ ]:
from pathlib import Path

data_dir = Path(
    "/content/drive/MyDrive/CET433_Explainable_IDS/data/official"
)

for filename in [
    "UNSW_NB15_training-set.csv",
    "UNSW_NB15_testing-set.csv",
]:
    path = data_dir / filename
    if path.is_file():
        print(f"✅ {filename}: {path.stat().st_size:,} bytes")
    else:
        print(f"❌ Missing: {filename}")

In [ ]:

from pathlib import Path
import hashlib

data_dir = Path(
    "/content/drive/MyDrive/CET433_Explainable_IDS/data/official"
)

expected_hashes = {
    "UNSW_NB15_training-set.csv":
        "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa",
    "UNSW_NB15_testing-set.csv":
        "734fe6642edf758f7c94d7d9149426b49d202fe8e7bf0bef47392489c3c0a559",
}

for filename, expected in expected_hashes.items():
    path = data_dir / filename

    sha256 = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    actual = sha256.hexdigest()

    print(f"\nFile: {filename}")
    print(f"SHA-256: {actual}")
    print("✅ VERIFIED" if actual == expected else "❌ HASH MISMATCH")


In [ ]:
from pathlib import Path
import hashlib

snapshot = Path(
    "/content/drive/MyDrive/CET433_Explainable_IDS/"
    "snapshots/CET433_colab_snapshot.zip"
)

expected = (
    "04972faa25f109c5a8d526771cd375796c421b8894df021f3889c00bca376776"
)

if not snapshot.is_file():
    print("❌ Snapshot not found")
else:
    actual = hashlib.sha256(snapshot.read_bytes()).hexdigest()

    print("Snapshot found:", snapshot.name)
    print("SHA-256:", actual)
    print("✅ VERIFIED" if actual == expected else "❌ HASH MISMATCH")

## M03.4 source bootstrap
For Stage B, run the source bootstrap cell only. The paired dataset verification cell reads both partitions. Follow the M03.4 runbook for training-only preflight.

In [ ]:
# M03.4 source bootstrap. Skip the paired dataset verification for Stage B.
import hashlib
import io
import json
import os
from pathlib import Path
import sys
import tempfile
from zipfile import ZipFile

try:
    from google.colab import drive
except ModuleNotFoundError as exc:
    if exc.name not in {"google", "google.colab"}:
        raise
    drive = None

ON_COLAB = drive is not None
SNAPSHOT_MAX_BYTES = 1024 * 1024
if ON_COLAB:
    drive.mount("/content/drive")
    snapshot = Path("/content/drive/MyDrive/CET433_Explainable_IDS/snapshots/CET433_colab_m03.4-stage-b.zip")
    expected_sha256 = os.environ.get("CET433_SOURCE_SHA256") or input("Local source snapshot SHA-256: ")
    if not snapshot.is_file():
        raise FileNotFoundError(f"Source snapshot missing: {snapshot}")
    with snapshot.open("rb") as source:
        payload = source.read(SNAPSHOT_MAX_BYTES + 1)
    if len(payload) > SNAPSHOT_MAX_BYTES:
        raise ValueError("Expected a source-only snapshot under 1 MiB.")
    if hashlib.sha256(payload).hexdigest() != expected_sha256.strip().lower():
        raise ValueError("Colab source snapshot SHA-256 does not match the locally recorded digest.")
    expected_files = {
        "CET433_Project2_IDS_M01R.ipynb",
        "colab_acceptance.py",
        "colab_preflight.py",
        "colab_snapshot.py",
        "data/dataset_manifest.json",
        "data/m02.2_split_manifest.json",
        "docs/m02.2-split-protocol.md",
        "docs/m02.3-colab-qualification.md",
        "docs/m02.4-m03-execution.md",
        "environment/colab-py313-v1.in",
        "environment/local-jupyter-py313-v1.in",
        "environment_smoke.py",
        "grouped_splits.py",
        "ids_utils.py",
        "model_artifact.py",
        "m03_runner.py",
        "requirements.txt",
        "stage_a.py",
        "stage_b.py",
    }
    with ZipFile(io.BytesIO(payload)) as archive:
        members = archive.infolist()
        if len(members) != len(expected_files) or {item.filename for item in members} != expected_files:
            raise ValueError("Unexpected source snapshot contents.")
        if sum(item.file_size for item in members) > SNAPSHOT_MAX_BYTES:
            raise ValueError("Expanded source snapshot exceeds 1 MiB.")
        PROJECT_DIR = Path(tempfile.mkdtemp(prefix="cet433-source-", dir="/content"))
        for name in sorted(expected_files):
            target = PROJECT_DIR / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(name))
    print("Source snapshot verified:", expected_sha256.strip().lower())
else:
    start = Path(os.environ.get("CET433_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
    PROJECT_DIR = next((p for p in (start, *start.parents) if (p / "colab_preflight.py").is_file()), None)
    if PROJECT_DIR is None:
        raise RuntimeError("Start in the CET433 checkout or set CET433_PROJECT_ROOT.")

for module_name in ("colab_preflight", "ids_utils", "model_artifact", "grouped_splits",
                    "colab_acceptance", "colab_snapshot"):
    cached = sys.modules.get(module_name)
    if cached is not None and Path(cached.__file__).resolve().parent != PROJECT_DIR.resolve():
        raise RuntimeError(f"{module_name} was loaded from another source. Restart the kernel.")
sys.path.insert(0, str(PROJECT_DIR))
from colab_preflight import dataset_directory, runtime_inventory, verify_datasets

print("Runtime:", "Colab" if ON_COLAB else "local")
profile = "environment/colab-py313-v1.in" if ON_COLAB else "requirements.txt"
print("Profile inventory:", profile)
print(json.dumps(runtime_inventory(PROJECT_DIR / profile), indent=2))


In [ ]:
# M02.0C preflight ends after this cell.
DATA_DIR = dataset_directory(
    PROJECT_DIR, cloud=ON_COLAB, override=os.environ.get("CET433_DATA_DIR")
)
DATASET_REPORT = verify_datasets(DATA_DIR, PROJECT_DIR / "data" / "dataset_manifest.json")
TRAIN_PATH = DATASET_REPORT["training"]["path"]
TEST_PATH = DATASET_REPORT["testing"]["path"]
print(json.dumps(DATASET_REPORT, indent=2))


## Model execution — controlled subprocess only
The reviewed cells below are retained for traceability. Use the M02.4 runner and leave the notebook model guard disabled.


In [ ]:
# Model execution is unavailable from the interactive notebook.
ALLOW_MODEL_EXECUTION = False
raise RuntimeError("M02.4 stops at preflight. Use the qualified isolated runner after approval.")


In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

df_raw = read_flow_csv(TRAIN_PATH)
print("Training file shape:", df_raw.shape)
df_raw.head()


## 3. Exploratory data analysis

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

print(df_raw.dtypes.value_counts(), "\n")
print("Duplicate rows (all columns except id):", df_raw.drop(columns=["id"], errors="ignore").duplicated().sum())
print("Missing values:", int(df_raw.isna().sum().sum()))
num_only = df_raw.select_dtypes(include=np.number)
print("Inf values:", int(np.isinf(num_only).sum().sum()))
df_raw.describe().T.head(15)

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df_raw["label"].map({0: "Normal", 1: "Attack"}).value_counts().plot.bar(ax=ax[0], color=["#4C78A8", "#E45756"])
ax[0].set_title("Binary class balance")
df_raw["attack_cat"].value_counts().plot.barh(ax=ax[1])
ax[1].set_title("Attack category counts"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()
print(df_raw["attack_cat"].value_counts())

## 4. Features and targets
Preserve every source observation and original target. Exclude the ID and both targets from the ordered model features.


In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

dataset_manifest = json.loads((PROJECT_DIR / "data/dataset_manifest.json").read_text(encoding="utf-8"))
feature_columns = model_feature_columns(dataset_manifest)
X = prepare_features(df_raw, feature_columns)
y_bin, y_multi = df_raw["label"], df_raw["attack_cat"]
print("Features:", X.shape[1])


## Frozen grouped assignments
The runner verifies the M02.2 assignment file against the training CSV and provides the frozen split plan in original row order.


In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

fit_pos, val_pos = split_plan.fit, split_plan.validation
X_train, X_val = X.iloc[fit_pos], X.iloc[val_pos]
yb_train, yb_val = y_bin.iloc[fit_pos], y_bin.iloc[val_pos]
ym_train, ym_val = y_multi.iloc[fit_pos], y_multi.iloc[val_pos]
print("Frozen split:", split_plan.assignment_sha256)


## 6. Preprocessing pipeline
One `Pipeline` per model. Everything that learns from data (median imputer, scaler, one-hot categories) is fitted **only on the training fold** because it lives inside the pipeline.

- Numeric: median impute -> `log1p` (heavy-tailed byte/packet/load features) -> standard scale
- Categorical (`proto`, `service`, `state`): one-hot, rare categories grouped, unseen categories ignored
- Class imbalance: `class_weight="balanced"` inside each classifier (training data only; SMOTE is a possible extension with an `imblearn` pipeline)

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

from ids_utils import CATEGORICAL_COLS, make_preprocessor

cat_cols = [name for name in feature_columns if name in CATEGORICAL_COLS]
num_cols = [name for name in feature_columns if name not in CATEGORICAL_COLS]
preprocess = make_preprocessor(feature_columns)
def make_pipe(clf):
    return Pipeline([("pre", clone(preprocess)), ("clf", clf)])

def rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return dict(precision=precision_score(y_true, y_pred, zero_division=0),
                recall=recall_score(y_true, y_pred), f1=f1_score(y_true, y_pred),
                fpr=float(fp / (fp + tn)), TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))
print("numeric:", len(num_cols), "| categorical:", cat_cols)

## 7. Leakage check (documented)
1. Features contain no target or identifier columns.
2. No row index and no identical feature row appears in both train and validation.
3. The scaler saw exactly the training rows (not validation).
4. Shuffled-label test: if labels are shuffled, ROC-AUC must fall to about 0.5. If it does not, something leaks.

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

# 1) forbidden columns
assert not {"label", "attack_cat", "id"} & set(X.columns), "target/ID column in features"

# 2) overlap
assert len(set(X_train.index) & set(X_val.index)) == 0, "index overlap"
assert set(split_plan.groups[fit_pos]).isdisjoint(split_plan.groups[val_pos]), "feature group overlap"

# 3) preprocessing fitted on train only
probe = make_pipe(DecisionTreeClassifier(random_state=SEED)).fit(X_train, yb_train)
n_seen = probe.named_steps["pre"].named_transformers_["num"].named_steps["sc"].n_samples_seen_
assert n_seen == len(X_train), "scaler saw more than the training rows"
print("Scaler fitted on", n_seen, "rows = len(X_train) OK")

# 4) shuffled-label sanity test (ROC-AUC, because F1 is inflated by the ~64% attack share)
rng = np.random.RandomState(SEED)
yb_shuffled = pd.Series(rng.permutation(yb_train.values), index=yb_train.index)
def val_auc(y_fit):
    m = make_pipe(DecisionTreeClassifier(max_depth=10, random_state=SEED)).fit(X_train, y_fit)
    return roc_auc_score(yb_val, m.predict_proba(X_val)[:, 1])
auc_shuf, auc_real = val_auc(yb_shuffled), val_auc(yb_train)
print(f"Validation ROC-AUC with shuffled labels: {auc_shuf:.3f} (expected ~0.5)  |  with real labels: {auc_real:.3f}")
assert auc_shuf < 0.6, "possible leakage: shuffled labels still score well"
assert auc_real - auc_shuf > 0.2, "real labels should beat shuffled labels clearly"
print("Leakage check passed")

## 8. Baseline models: binary task (Normal vs Attack)
Metrics reported: confusion matrix, precision, recall, F1 and False Positive Rate. **Accuracy alone is not used**: a SOC cares about missed attacks (recall) and about alert fatigue (false positives).

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

def eval_binary(name, pipe):
    pipe.fit(X_train, yb_train)
    pred = pipe.predict(X_val)
    tn, fp, fn, tp = confusion_matrix(yb_val, pred).ravel()
    res = dict(model=name,
               precision=precision_score(yb_val, pred),
               recall=recall_score(yb_val, pred),
               f1=f1_score(yb_val, pred),
               fpr=fp / (fp + tn), TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))
    ConfusionMatrixDisplay(confusion_matrix(yb_val, pred), display_labels=["Normal", "Attack"]).plot(cmap="Blues")
    plt.title(f"{name}: binary confusion matrix"); plt.show()
    return res

baselines = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
}
bin_results = [eval_binary(n, make_pipe(m)) for n, m in baselines.items()]
bin_table = pd.DataFrame(bin_results).set_index("model")
bin_table.round(4)

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

target_recall, target_fpr = 0.95, 0.01
for name, r in bin_table.iterrows():
    ok = r["recall"] >= target_recall and r["fpr"] < target_fpr
    print(f"{name:20s} recall={r['recall']:.3f} fpr={r['fpr']:.3f} -> success metric {'MET' if ok else 'NOT met (notebook 2 aims to improve it)'}")

## 9. Baseline model: multiclass task (attack family)
Per-class recall shows which attack families the baseline misses; rare classes (Worms, Shellcode, Backdoor, Analysis) are the usual weak spots and the focus of the final models.

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

multi_pipe = make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))
multi_pipe.fit(X_train, ym_train)
pm = multi_pipe.predict(X_val)

print(classification_report(ym_val, pm, digits=3, zero_division=0))
labels = sorted(ym_val.unique())
cm = confusion_matrix(ym_val, pm, labels=labels, normalize="true")
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Logistic Regression baseline: row-normalised confusion matrix (diagonal = per-class recall)")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.show()

per_class_recall = pd.Series(recall_score(ym_val, pm, labels=labels, average=None, zero_division=0), index=labels)
per_class_recall.round(3)

## 10. Baselines with latency and ROC-AUC
The same two baselines, now with ROC-AUC and latency per prediction, so they can be compared with the tuned models in one table.

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

baseline_pipes = {
    "Logistic Regression (baseline)": make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
    "KNN k=5 (baseline)": make_pipe(KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
}
def latency_ms(pipe, X_ref, n=200):
    # time to score ONE flow at a time, which is what a live alert stream needs
    rows = [X_ref.iloc[[i]] for i in range(min(n, len(X_ref)))]
    pipe.predict_proba(rows[0])  # warm-up
    t0 = time.perf_counter()
    for r in rows:
        pipe.predict_proba(r)
    return (time.perf_counter() - t0) / len(rows) * 1000

results = {}
fitted = {}
for name, pipe in baseline_pipes.items():
    pipe.fit(X_train, yb_train)
    proba = pipe.predict_proba(X_val)[:, 1]
    r = rates(yb_val, (proba >= 0.5).astype(int))
    r.update(roc_auc=roc_auc_score(yb_val, proba), latency_ms=latency_ms(pipe, X_val))
    results[name], fitted[name] = r, pipe
pd.DataFrame(results).T[["precision", "recall", "f1", "fpr", "roc_auc", "latency_ms"]].round(4)

## 11. Tuning with five group-disjoint CV folds
Precomputed training-only grouped assignments are passed directly to the search. Use all internal fit rows to avoid splitting feature groups during sampling. MLP is deferred because its own early-stopping holdout does not accept groups.


In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

cv = split_plan.cv()
X_tune, y_tune = X_train, yb_train
print("tuning rows:", len(X_tune))

search_space = {
    "Random Forest": (RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=SEED),
        {"clf__n_estimators": [100, 200], "clf__max_depth": [None, 15, 25],
         "clf__min_samples_leaf": [1, 2, 5]}),
    "LightGBM": (LGBMClassifier(class_weight="balanced", random_state=SEED, verbose=-1, n_jobs=-1),
        {"clf__n_estimators": [200, 400], "clf__learning_rate": [0.05, 0.1],
         "clf__num_leaves": [31, 63], "clf__min_child_samples": [20, 50]}),

}
best_params, cv_scores = {}, {}
for name, (clf, grid) in search_space.items():
    s = RandomizedSearchCV(make_pipe(clf), grid, n_iter=6, cv=cv, scoring="f1",
                           random_state=SEED, n_jobs=1, refit=False)
    s.fit(X_tune, y_tune)
    best_params[name], cv_scores[name] = s.best_params_, s.best_score_
    print(f"{name:14s} best CV F1 = {s.best_score_:.4f}  {s.best_params_}")

## 12. Final fit and model comparison (validation split)

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

tuned = {}
for name, (clf, _) in search_space.items():
    pipe = make_pipe(clone(clf)).set_params(**best_params[name])
    pipe.fit(X_train, yb_train)
    tuned[name] = pipe
    proba = pipe.predict_proba(X_val)[:, 1]
    r = rates(yb_val, (proba >= 0.5).astype(int))
    r.update(roc_auc=roc_auc_score(yb_val, proba), latency_ms=latency_ms(pipe, X_val), cv_f1=cv_scores[name])
    results[name], fitted[name] = r, pipe

comp = pd.DataFrame(results).T[["precision", "recall", "f1", "fpr", "roc_auc", "latency_ms", "cv_f1"]].astype(float)
best_base_f1 = comp.loc[[n for n in comp.index if "baseline" in n], "f1"].max()
comp["f1_gain_vs_best_baseline"] = comp["f1"] - best_base_f1
comp.round(4)

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
comp["f1"].plot.barh(ax=ax[0], color="#4C78A8"); ax[0].set_title("F1 on validation (threshold 0.5)")
comp["latency_ms"].plot.barh(ax=ax[1], color="#E45756"); ax[1].set_title("Latency per prediction (ms, one flow at a time)")
plt.tight_layout(); plt.show()

### Deployment model
Choose between Random Forest and LightGBM by validation F1. The official test partition remains outside this decision.


In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

tree_names = ["Random Forest", "LightGBM"]
final_name = comp.loc[tree_names, "f1"].idxmax()
final_pipe = tuned[final_name]
proba_val = final_pipe.predict_proba(X_val)[:, 1]
print("Deployment model:", final_name)

## 13. Validation-only operating threshold
Choose the lowest estimated cost among thresholds with recall ≥95% and FPR <1%. If infeasible, disclose that outcome and use the minimum-cost validation threshold for a descriptive benchmark.


In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

COST_FP_MIN = 5
COST_FN_MIN = 120

def total_cost_min(fp, fn):
    return fp * COST_FP_MIN + fn * COST_FN_MIN

# Examine every distinct validation decision boundary, including domain endpoints.
order = np.argsort(proba_val)
sorted_scores = proba_val[order]
thresholds = np.unique(np.r_[0.0, sorted_scores, 1.0])
starts = np.searchsorted(sorted_scores, thresholds)
positives = np.r_[0, np.cumsum(yb_val.to_numpy()[order])]
fn = positives[starts]
negatives = len(sorted_scores) - positives[-1]
fp = negatives - (starts - fn)
curve = pd.DataFrame({
    "threshold": thresholds,
    "recall": 1 - fn / positives[-1],
    "fpr": fp / negatives,
    "cost_min": total_cost_min(fp, fn),
})

feasible = curve[(curve["recall"] >= 0.95) & (curve["fpr"] < 0.01)]
joint_feasible = not feasible.empty
candidates = feasible if joint_feasible else curve
# Break equal-cost ties by lower FPR and then higher threshold.
selected = candidates.sort_values(["cost_min", "fpr", "threshold"], ascending=[True, True, False]).iloc[0]
THRESHOLD = float(selected["threshold"])
print("Joint validation target feasible:", joint_feasible, "| frozen threshold:", THRESHOLD)

plt.figure(figsize=(7, 4))
plt.plot(curve["threshold"], curve["cost_min"] / 60)
plt.axvline(THRESHOLD, ls="--", c="C3", label=f"selected = {THRESHOLD}")
plt.xlabel("Alert threshold"); plt.ylabel("Analyst hours (validation set)"); plt.legend()
plt.title("Cost curve"); plt.show()

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

final = rates(yb_val, (proba_val >= THRESHOLD).astype(int))
print({k: round(v, 4) if isinstance(v, float) else v for k, v in final.items()})
print("Success metric (recall >= 95% and FPR < 1%):",
      "MET" if final["recall"] >= 0.95 and final["fpr"] < 0.01 else "NOT met at this threshold")

base_name = max([n for n in results if "baseline" in n], key=lambda n: results[n]["f1"])
base_cost = total_cost_min(results[base_name]["FP"], results[base_name]["FN"])
new_cost = total_cost_min(final["FP"], final["FN"])
hours_saved = (base_cost - new_cost) / 60
per_100k = hours_saved / len(X_val) * 100000
print(f"Analyst hours saved vs {base_name}: {hours_saved:.1f} h on {len(X_val)} flows (about {per_100k:.0f} h per 100,000 flows)")

## 14. Explainability with SHAP
Global view first (which features drive alerts), then three individual alerts: a true positive, a **false negative** and a **false positive**, as required.

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

pre, clf = final_pipe.named_steps["pre"], final_pipe.named_steps["clf"]
feat_names = list(pre.get_feature_names_out())
explainer = shap.TreeExplainer(clf)

def shap_attack(Xt):
    # normalise the different SHAP output shapes to values for the "attack" class
    sv = explainer.shap_values(Xt)
    if isinstance(sv, list):
        sv = sv[1]
    sv = np.asarray(sv)
    return sv[:, :, 1] if sv.ndim == 3 else sv

def base_attack():
    ev = np.atleast_1d(explainer.expected_value)
    return float(ev[1] if len(ev) > 1 else ev[0])

n_sample = min(2000, len(X_val))
samp = X_val.sample(n_sample, random_state=SEED)
Xt_samp = pre.transform(samp)
sv_samp = shap_attack(Xt_samp)
shap.summary_plot(sv_samp, Xt_samp, feature_names=feat_names, max_display=15, show=False)
plt.title(f"SHAP summary: what pushes a flow towards 'attack' ({final_name})")
plt.tight_layout(); plt.show()

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

pred_val = (proba_val >= THRESHOLD).astype(int)

def pick(mask, ascending):
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return None
    return int(idx[np.argsort(proba_val[idx])[0 if ascending else -1]])

cases = {
    "False negative (attack missed)": pick((yb_val.values == 1) & (pred_val == 0), ascending=True),
    "False positive (false alarm)": pick((yb_val.values == 0) & (pred_val == 1), ascending=False),
}
if cases["False negative (attack missed)"] is None:   # no misses at this threshold: show the attack the model was least sure about
    cases["False negative (attack missed)"] = pick(yb_val.values == 1, ascending=True)
    print("No false negatives at this threshold; showing the attack with the lowest score instead.")
if cases["False positive (false alarm)"] is None:
    cases["False positive (false alarm)"] = pick(yb_val.values == 0, ascending=False)
    print("No false positives at this threshold; showing the normal flow with the highest score instead.")

def top_reasons(i, k=6):
    xt = pre.transform(X_val.iloc[[i]])
    sv = shap_attack(xt)[0]
    order = np.argsort(-np.abs(sv))[:k]
    return pd.DataFrame({"feature": [feat_names[j] for j in order],
                         "value_after_preprocessing": np.round(xt[0, order], 2),
                         "shap": np.round(sv[order], 3),
                         "pushes_towards": np.where(sv[order] > 0, "attack", "normal")}), xt, sv

for title, i in cases.items():
    tbl, xt, sv = top_reasons(i)
    print(f"\n=== {title}: flow #{i} | true class: {ym_val.iloc[i]} | model score: {proba_val[i]:.3f} | threshold: {THRESHOLD} ===")
    display(tbl)
    shap.plots.waterfall(shap.Explanation(values=sv, base_values=base_attack(), data=xt[0],
                                          feature_names=feat_names), max_display=10, show=False)
    plt.title(title); plt.tight_layout(); plt.show()

**How to read these for the report:** a false negative is an attack whose strongest features looked normal to the model (explain which features pulled the score down and why that attack type resembles normal traffic). A false positive is normal traffic that looked like an attack on its strongest features (for example unusual packet counts or rates). Write one short paragraph per case, using the table above.

## 15. Multiclass view (attack family)
LightGBM with the tuned settings versus the Logistic Regression baseline, compared by macro-F1 and per-class recall.

In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

lgb_params = {k: v for k, v in best_params["LightGBM"].items()}
multi_lgbm = make_pipe(LGBMClassifier(class_weight="balanced", random_state=SEED, verbose=-1, n_jobs=-1)).set_params(**lgb_params)
multi_lgbm.fit(X_train, ym_train)
multi_lr = make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)).fit(X_train, ym_train)

labels = sorted(ym_val.unique())
rec = pd.DataFrame({
    "LogReg baseline": recall_score(ym_val, multi_lr.predict(X_val), labels=labels, average=None, zero_division=0),
    "LightGBM": recall_score(ym_val, multi_lgbm.predict(X_val), labels=labels, average=None, zero_division=0)}, index=labels)
print("macro-F1  LogReg:", round(f1_score(ym_val, multi_lr.predict(X_val), average="macro"), 4),
      "| LightGBM:", round(f1_score(ym_val, multi_lgbm.predict(X_val), average="macro"), 4))
rec.round(3)

## 16. Final official test evaluation — protocol cet433-eval-v1
Use the frozen validation threshold. Report all original official test observations and the exact-feature-unseen subset relative to final model fit rows. Report support and prevalence per stratum. This is a held-out benchmark without temporal claims.


In [ ]:
if globals().get("CET433_CONTROLLED_RUN") is not True:
    raise RuntimeError("Use the controlled CET433 runner.")

df_test = read_flow_csv(TEST_PATH)
X_test = prepare_features(df_test, feature_columns)
yb_test, ym_test = df_test["label"], df_test["attack_cat"]
# Resolve exact equality across fit and test in the same fingerprint call.
combined_groups = feature_groups(pd.concat([X_train, X_test], ignore_index=True), feature_columns)
unseen = ~pd.Index(combined_groups[len(X_train):]).isin(combined_groups[:len(X_train)])
proba_test = final_pipe.predict_proba(X_test)[:, 1]
pt = multi_lgbm.predict(X_test)
labels = sorted(y_multi.unique())

def evaluate_stratum(mask):
    binary, family = yb_test[mask], ym_test[mask]
    result = {
        "rows": len(binary),
        "binary_support": binary.value_counts().sort_index().to_dict(),
        "attack_prevalence": float(binary.mean()) if len(binary) else None,
        "attack_category_support": family.value_counts().sort_index().to_dict(),
        "binary_metrics": None,
        "per_class_recall": {},
    }
    if binary.nunique() == 2:
        scores = proba_test[mask]
        result["binary_metrics"] = {
            **rates(binary, (scores >= THRESHOLD).astype(int)),
            "roc_auc": roc_auc_score(binary, scores),
        }
    if len(family):
        for category in labels:
            rows = family.to_numpy() == category
            result["per_class_recall"][category] = float(np.mean(pt[mask][rows] == category)) if rows.any() else None
    return result

test_report = {
    "official_raw": evaluate_stratum(np.ones(len(X_test), dtype=bool)),
    "exact_feature_unseen": evaluate_stratum(unseen),
}
display(test_report)


## Persistent artifacts
The isolated runner writes model artifacts, frozen selection, metrics, and provenance to a new Drive run directory.


Artifacts are exported by the controlled runner. Explainability cells are outside its M03 execution plan.


## 18. Required results to report

Fill in these points from the executed results:

- Best tuned model and its improvement over the baseline.
- Precision, recall, F1, confusion matrix, and latency.
- Per-class recall for the multiclass task.
- Whether the stated business target was achieved; do not claim success if the measured results do not meet it.
- Analyst-time business metric and its assumptions.
- SHAP explanation for one false negative and one false positive.
- Difference between validation and official unseen-test results, with the temporal-split limitation stated clearly.


**Dashboard:** The companion Streamlit app is provided as `app/soc_dashboard.py` in the project bundle. Run it after generating `models/ids_model.joblib`.

**Temporal split:** The supplied UNSW-NB15 partition CSVs do not expose a reliable timestamp column. A real chronological comparison cannot be computed from these files without another timestamped source. `temporal_split_helper.py` is provided to perform the split once a real timestamp field is available; it intentionally does not fabricate dates.
